# Gold Layer — Retail Logistics KPIs

## Objective

Transform validated Silver shipment records into
business-level delivery performance metrics.

**Source:** workspace.silver.shipments

**Target:** workspace.gold.delivery_kpis

**Processing strategy:** Full refresh

**Technologies:** Databricks, Spark SQL, Delta Lake

### 3.1. Configuration

In [0]:
SOURCE_TABLE = "workspace.silver.shipments"
TARGET_TABLE = "workspace.gold.delivery_kpis"

###3.2. Create the Gold schema

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

### 3.2. Build Gold KPIs

In [0]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {TARGET_TABLE}
    USING DELTA
    AS
    SELECT
        distribution_center,
        COUNT(*) AS total_shipments,

        COUNT_IF(
            delivery_timeliness = 'ON_TIME'
        ) AS on_time_shipments,

        COUNT_IF(
            delivery_timeliness = 'LATE'
        ) AS late_shipments,

        COUNT_IF(
            fulfillment_status = 'UNDER_DELIVERY'
        ) AS under_deliveries,

        ROUND(
            100.0 * COUNT_IF(
                delivery_timeliness = 'ON_TIME'
            ) / COUNT(*),
            2
        ) AS on_time_delivery_pct,

        ROUND(
            100.0 * COUNT_IF(
                fulfillment_status = 'UNDER_DELIVERY'
            ) / COUNT(*),
            2
        ) AS under_delivery_pct

    FROM {SOURCE_TABLE}
    GROUP BY distribution_center
""")

print(f"Gold table created: {TARGET_TABLE}")

###3.4 Display the resulting Gold data

In [0]:
gold_df = spark.table(TARGET_TABLE)

display(gold_df)